In [1]:
!pip install boto3


In [2]:
import boto3
import os

In [3]:
from pathlib import Path

aws_access_key_id ="AKIA2JHUK4EGBVSQ5RUWssss"
aws_secret_access_key = "6os7o+kr8eVGS1Mqxrvo57UPlhFY3Yag9IDswbc4"

s3_client = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
bucket_name ="anyoneai-datasets"
prefix = "queplan_insurance/"  

files= []
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
files =[]
if "Contents" in response:
    files = [obj["Key"] for obj in response["Contents"]]
    print("Archivos disponibles:", files)
else:
    print("No se encontraron archivos en el dataset.")
    
download_path =  os.path.join("./docs/dataset/")
for file in files:
    document=file.split("/")[-1]
    document_path=os.path.join(download_path,document)

    if not os.path.isfile(document_path) and document_path.endswith(".pdf"):
        file_name = os.path.join(download_path, document)
        print(file_name)
        s3_client.download_file(bucket_name, file, file_name)
        print(f"Descargado: {file}")

ClientError: An error occurred (InvalidAccessKeyId) when calling the ListObjectsV2 operation: The AWS Access Key Id you provided does not exist in our records.

In [ ]:
### Esto es para fusionar tanto la base de datos de los PDFs como de nuestro documento de Q&A
from langchain.vectorstores import FAISS

# Cargamos la base de datos FAISS
vector_db_policies = FAISS.load_local("insurance_policies_db", embedding_model, allow_dangerous_deserialization=True,)
# Cargarmos la base de datos de Q&A propia
vector_db_qna = FAISS.load_local("insurance_qna_db", embedding_model, allow_dangerous_deserialization=True)

# Combinamos ambas bases FAISS en una sola
vector_db_policies.merge_from(vector_db_qna)

# Ahora vector_db contiene ambos conjuntos de datos
vector_db = vector_db_policies

print("✅ FAISS con documentos y Q&A cargado correctamente.")

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.
`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


✅ FAISS con documentos y Q&A cargado correctamente.


In [ ]:
### Aquí generamos los tokens a partir del modelo descargado
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

#model_path = "./llama3_3b"  # Ruta donde descargaste el modelo
# model_path = "./microsoft_phi-2"  # Ruta donde descargaste el modelo
model_path = "./zephyr-7b-alpha"  # Ruta donde descargaste el modelo
model_path
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

OSError: Incorrect path_or_model_id: './zephyr-7b-alpha'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [9]:
def retrieve_relevant_docs(query, top_k=3):
    query_embedding = embedding_model.encode(query)
    retrieved_docs = vector_db.similarity_search_by_vector(query_embedding, k=top_k)
    return retrieved_docs

In [10]:
def format_prompt(query):
    relevant_docs = retrieve_relevant_docs(query)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    prompt = f"""Usa la siguiente información para responder de manera clara y precisa:

    {context}

    Pregunta del usuario: {query}
    """
    
    return prompt


In [ ]:
### Integración de SerpAPI GOOGLE Search OPCIONAL
#from langchain_community.tools import SerpAPIWrapper
from langchain_community.utilities import SerpAPIWrapper

search = SerpAPIWrapper(serpapi_api_key="tu clave")  # Obtén la clave en https://serpapi.com/

def search_web(query, num_results=3):
    results = search.run(query)
    return results[:num_results]

query = "Últimas noticias sobre seguros en América"
results = search_web(query)
print(results)

[' 


In [12]:
### Prompt integrando búsquedas con GOOGLE
def format_prompt(query):
    relevant_docs = retrieve_relevant_docs(query)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # Buscamos en Google si no hay documentos relevantes
    if not context:
        context = search_web(query)

    prompt = f"""Usa la siguiente información para responder de manera clara y precisa:

    {context}

    Pregunta del usuario: {query}
    """

    return prompt

In [9]:
### Aquí configuramos el texto por respuesta, temperatura, etc, a partir de nuestro modelo
import torch

def generate_response(query):
    prompt = format_prompt(query)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    model.to(inputs.input_ids.device)
    
    output = model.generate(**inputs, max_length=500, temperature=0.7)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    return response

In [10]:
### Aquí está la primera pregunta de nuestro PROMPT al RAG creado
query = "¿Qué se considera un Accidente según la póliza POL120190177?"
response = generate_response(query)
print(response)

NameError: name 'tokenizer' is not defined

In [ ]:
query = "¿Me podrías crear una póliza de seguro de hogar?"
response = generate_response(query)
print(response)

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from ragas.evaluation import evaluate
from ragas.metrics import context_precision, context_recall
from datasets import Dataset

# Cargamos el modelo y tokenizador de Hugging Face
model_id = "unsloth/Llama-3.2-3B" # Aquí el nombre del modelo que estén usando
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Creamos el pipeline de generación de texto
text_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Integramos el modelo en LangChain
llm = HuggingFacePipeline(pipeline=text_pipeline)

# Aquí va nuestro feedback para medir y mejorar la precisión del modelo
data = Dataset.from_dict({
    "question": [query],
    "answer": [response],
    "contexts": [[docs[0].page_content]],
    "reference": ["Un Accidente es un suceso imprevisto, involuntario, repentino y fortuito, causado por medios externos y de modo violento, que afecta al organismo del asegurado, ocasionándole lesiones visibles o internas."]
})

# Evaluar con modelo de Hugging Face
#evaluation = evaluate(data, metrics=[context_precision, context_recall], llm=llm)
evaluation = evaluate(data, metrics=[context_precision, context_recall], llm=llm, batch_size=1)
print("\n📊 Evaluación de la respuesta:", evaluation)

Device set to use cpu
C:\Users\AER\AppData\Local\Temp\ipykernel_5940\2555293172.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_pipeline)


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Batch 1/2:   0%|          | 0/1 [00:00<?, ?it/s]

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()



📊 Evaluación de la respuesta: {'context_precision': nan, 'context_recall': nan}


In [ ]:
import ragas.metrics
print(dir(ragas.metrics))

['AgentGoalAccuracyWithReference', 'AgentGoalAccuracyWithoutReference', 'AnswerCorrectness', 'AnswerRelevancy', 'AnswerSimilarity', 'AspectCritic', 'BleuScore', 'ContextEntityRecall', 'ContextPrecision', 'ContextRecall', 'ContextUtilization', 'DataCompyScore', 'DistanceMeasure', 'ExactMatch', 'FactualCorrectness', 'Faithfulness', 'FaithfulnesswithHHEM', 'InstanceRubrics', 'LLMContextPrecisionWithReference', 'LLMContextPrecisionWithoutReference', 'LLMContextRecall', 'LLMSQLEquivalence', 'Metric', 'MetricOutputType', 'MetricType', 'MetricWithEmbeddings', 'MetricWithLLM', 'MultiModalFaithfulness', 'MultiModalRelevance', 'MultiTurnMetric', 'NoiseSensitivity', 'NonLLMContextPrecisionWithReference', 'NonLLMContextRecall', 'NonLLMStringSimilarity', 'ResponseRelevancy', 'RougeScore', 'RubricsScore', 'SemanticSimilarity', 'SimpleCriteriaScore', 'SingleTurnMetric', 'StringPresence', 'SummarizationScore', 'ToolCallAccuracy', 'TopicAdherenceScore', '__all__', '__builtins__', '__cached__', '__doc__

In [ ]:
query = "¿Qué gastos no son cubiertos por la póliza POL120190177?"
response = generate_response(query)
print(response)

Usa la siguiente información para responder de manera clara y precisa:

    ocurridos durante el mismo año póliza. Al concretarse la renovación de la póliza, se establecerá una nueva
suma asegurada por asegurado, por año póliza, para los gastos incurridos por accidentes, enfermedades o
padecimientos cubiertos por la renovación en curso, así como a los gastos incurridos en esta nueva
vigencia, por accidentes, enfermedades o padecimientos cubiertos en las vigencias previas, aún para

Póliza, que, habiendo superado el Deducible, la compañía reembolsará al Asegurado Titular o, en su
defecto, a los herederos legales de éste, o pagará al Prestador, los Gastos Reembolsables durante la
vigencia de este contrato de seguro y en los términos y condiciones señalados en estas Condiciones
Generales, todo lo que, por su naturaleza, se indica en las Condiciones Particulares de la Póliza.

Póliza, que, habiendo superado el Deducible, la compañía reembolsará al Asegurado Titular o en su defecto,
a los h